# PRAGMA Finetuning and Evaluation

This notebook demonstrates loading the pre-trained PRAGMA representations and fine-tuning the model on downstream tasks (e.g. Fraud Detection, Churn Prediction).

In [2]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
drive_project_root = Path('/content/drive/My Drive/finai-research')
os.chdir(drive_project_root)
sys.path.append(str(drive_project_root / 'src'))

Mounted at /content/drive


In [3]:

import torch
import torch.nn as nn
from pragma_model import PRAGMA

# 1. Reuse the EXACT configurations from our real Dataset
profile_config = {
    'num_numerical_features': 5,   # Match: C1, C2, C3, C4, C5
    'num_categorical_features': 3, # Match: card4, card6, P_emaildomain
    'cat_embedding_dims': [(100, 8), (100, 8), (100, 8)]
}

event_config = {
    'event_dim': 20, # Match: TransactionAmt + V1-V19
    'num_heads': 4,
    'num_layers': 3,
    'max_seq_len': 100
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Initialize Model (Notice num_classes=2 for Fraud vs. Not Fraud)
model = PRAGMA(profile_config, event_config, embed_dim=64, num_classes=2)

# 3. Load the Pretrained Weights
# FIXED PATH: Changed '../models/' to './models/'
checkpoint_path = './models/pragma_pretrained.pth'
try:
    # strict=False is important here because the pre-trained model didn't have
    # the final fraud classification layer, but this new one does!
    model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=False)
    print(f"SUCCESS: Loaded pre-trained weights from {checkpoint_path}!")
except Exception as e:
    print(f"ERROR: Could not load weights. {e}")

model.to(device)

SUCCESS: Loaded pre-trained weights from ./models/pragma_pretrained.pth!


PRAGMA(
  (profile_encoder): ProfileEncoder(
    (cat_embeddings): ModuleList(
      (0-2): 3 x Embedding(100, 8)
    )
    (mlp): Sequential(
      (0): Linear(in_features=29, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
      (3): Linear(in_features=128, out_features=64, bias=True)
    )
  )
  (event_encoder): EventEncoder(
    (event_embedding): Linear(in_features=20, out_features=64, bias=True)
    (positional_encoding): Embedding(100, 64)
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05,

In [4]:
from torch.utils.data import DataLoader
from data.dataset import IEEEDataset

print("Loading datasets...")
# Train dataset
train_dataset = IEEEDataset('data/processed/train.parquet', max_seq_len=100)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Validation dataset (make sure make_dataset.py actually generated this!)
val_dataset = IEEEDataset('data/processed/val.parquet', max_seq_len=100)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Datasets loaded successfully!")

Loading datasets...
Loading data from data/processed/train.parquet...
Grouping by ClientID...
Loading data from data/processed/val.parquet...
Grouping by ClientID...
Datasets loaded successfully!


## Finetuning Loop

In [5]:
from sklearn.metrics import roc_auc_score

def finetune_step(model, optimizer, criterion, batch):
    model.train()
    x_num, x_cat, events, seq_lengths, labels = batch

    optimizer.zero_grad()

    # Forward pass without pretrain flag
    logits = model(x_num, x_cat, events, seq_lengths, pretrain=False)

    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()

    return loss.item()

def evaluate(model, val_loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            # Send batch to device!
            x_num, x_cat, events, seq_lengths, labels = batch
            logits = model(x_num, x_cat, events, seq_lengths, pretrain=False)
            preds = torch.softmax(logits, dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    # ROC-AUC handles imbalanced fraud data perfectly
    return roc_auc_score(all_labels, all_preds)

# Setup Optimizer and criterion for classification
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [6]:
from sklearn.metrics import roc_auc_score

# --- THE ACTUAL TRAINING LOOP (Replaces the torch.randn block) ---
num_epochs = 5
print("Starting Real Fine-Tuning on IEEE Data...")

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        # Move batch to GPU
        batch = [b.to(device) for b in batch]

        loss = finetune_step(model, optimizer, criterion, batch)
        total_loss += loss

        if batch_idx % 20 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx}], Loss: {loss:.4f}")

    print("Evaluating on Validation Set...")
    val_auc = evaluate(model, val_loader)
    print(f"--> Epoch {epoch+1} Completed | Train Loss: {total_loss/(batch_idx+1):.4f} | Val ROC-AUC: {val_auc:.4f}\n")

Starting Real Fine-Tuning on IEEE Data...
Epoch [1/5], Step [0], Loss: 0.8651
Epoch [1/5], Step [20], Loss: 0.4953
Epoch [1/5], Step [40], Loss: 0.2510
Epoch [1/5], Step [60], Loss: 0.0733
Epoch [1/5], Step [80], Loss: 0.0287
Epoch [1/5], Step [100], Loss: 0.0225
Epoch [1/5], Step [120], Loss: 0.0220
Epoch [1/5], Step [140], Loss: 0.1318
Epoch [1/5], Step [160], Loss: 0.2336
Epoch [1/5], Step [180], Loss: 0.0314
Epoch [1/5], Step [200], Loss: 0.2361
Epoch [1/5], Step [220], Loss: 0.0275
Epoch [1/5], Step [240], Loss: 0.2299
Epoch [1/5], Step [260], Loss: 0.0368
Epoch [1/5], Step [280], Loss: 0.1264
Epoch [1/5], Step [300], Loss: 0.0239
Epoch [1/5], Step [320], Loss: 0.1460
Epoch [1/5], Step [340], Loss: 0.0398
Epoch [1/5], Step [360], Loss: 0.0209
Epoch [1/5], Step [380], Loss: 0.0240
Epoch [1/5], Step [400], Loss: 0.0128
Epoch [1/5], Step [420], Loss: 0.3415
Epoch [1/5], Step [440], Loss: 0.0370
Epoch [1/5], Step [460], Loss: 0.0266
Epoch [1/5], Step [480], Loss: 0.0259
Epoch [1/5], S

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


--> Epoch 1 Completed | Train Loss: 0.1249 | Val ROC-AUC: 0.7616

Epoch [2/5], Step [0], Loss: 0.1921
Epoch [2/5], Step [20], Loss: 0.0448
Epoch [2/5], Step [40], Loss: 0.2431
Epoch [2/5], Step [60], Loss: 0.1139
Epoch [2/5], Step [80], Loss: 0.1116
Epoch [2/5], Step [100], Loss: 0.0202
Epoch [2/5], Step [120], Loss: 0.0244
Epoch [2/5], Step [140], Loss: 0.0294
Epoch [2/5], Step [160], Loss: 0.2437
Epoch [2/5], Step [180], Loss: 0.2279
Epoch [2/5], Step [200], Loss: 0.4073
Epoch [2/5], Step [220], Loss: 0.1304
Epoch [2/5], Step [240], Loss: 0.1139
Epoch [2/5], Step [260], Loss: 0.0242
Epoch [2/5], Step [280], Loss: 0.0316
Epoch [2/5], Step [300], Loss: 0.1170
Epoch [2/5], Step [320], Loss: 0.0251
Epoch [2/5], Step [340], Loss: 0.2720
Epoch [2/5], Step [360], Loss: 0.1893
Epoch [2/5], Step [380], Loss: 0.0304
Epoch [2/5], Step [400], Loss: 0.2511
Epoch [2/5], Step [420], Loss: 0.0178
Epoch [2/5], Step [440], Loss: 0.0245
Epoch [2/5], Step [460], Loss: 0.0246
Epoch [2/5], Step [480], Los

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("Loading Test Dataset...")
test_dataset = IEEEDataset('data/processed/test.parquet', max_seq_len=100)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def evaluate_test(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            x_num, x_cat, events, seq_lengths, labels = [b.to(device) for b in batch]
            logits = model(x_num, x_cat, events, seq_lengths, pretrain=False)
            preds = torch.softmax(logits, dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate both metrics required for the comparison table
    roc_auc = roc_auc_score(all_labels, all_preds)
    pr_auc = average_precision_score(all_labels, all_preds)
    return roc_auc, pr_auc

print("Evaluating PRAGMA on Test Set...")
test_roc_auc, test_pr_auc = evaluate_test(model, test_loader)
print(f"✅ PRAGMA Test AUC-ROC: {test_roc_auc:.4f}")
print(f"✅ PRAGMA Test PR-AUC:  {test_pr_auc:.4f}")

# Save this fine-tuned model for the future!
torch.save(model.state_dict(), './models/pragma_finetuned.pth')

Loading Test Dataset...
Loading data from data/processed/test.parquet...
Grouping by ClientID...
Evaluating PRAGMA on Test Set...
✅ PRAGMA Test AUC-ROC: 0.7939
✅ PRAGMA Test PR-AUC:  0.1426
